In [ ]:
import copy
import csv
import os
import warnings
from argparse import ArgumentParser
import numpy as np
import torch
from tqdm import tqdm
import yaml
from torch.utils import data
# 개별 json 라벨 파일을 이용해 학습 데이터 리스트 생성
import glob
import json
import os
from nets import nn
from utils import util
import pandas as pd
from utils.dataset import Dataset
from PIL import Image
from torch.utils import data
import numpy as np
import cv2
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.model_selection import train_test_split
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:",device)
with open('utils/detail_args.yaml', errors='ignore') as f:
    params = yaml.safe_load(f) 

In [ ]:
input_size = 512

# ── 경로 설정 (IHC HER2) ────────────────────────────────────────
label_dir = '../../data/precise_BC_cell_scoring/her2/labels/'
image_dir = '../../data/precise_BC_cell_scoring/her2/patch_images/'
label_files = sorted(glob.glob(os.path.join(label_dir, '*.json')))

# ── 5 클래스 정의 (HER2 IHC scoring) ────────────────────────────
class_names = {
    0: "class0",   # 0+
    1: "class1",   # 1+
    2: "class2",   # 2+
    3: "class3",   # 3+
    4: "other"
}
num_classes = len(class_names)

# params['names'] 덮어쓰기
params['names'] = class_names

# ── 라벨 로드 (JSON: [{"class_id", "cx", "cy", "w", "h", "was_nonT"}, ...]) ──
# was_nonT 무시, 모든 세포 사용
image_filenames = []
labels = []   # 각 항목: [[class_id, y_top, x_left, h, w], ...] pixel 좌표

print("📂 Loading labels...")
for label_file in tqdm(label_files):
    base_name = os.path.splitext(os.path.basename(label_file))[0]
    img_path = os.path.join(image_dir, base_name + '.png')
    if not os.path.exists(img_path):
        continue

    with open(label_file) as f:
        data_json = json.load(f)

    boxes = []
    for ann in data_json:
        cls_id = int(ann['class_id'])
        cx, cy, w, h = ann['cx'], ann['cy'], ann['w'], ann['h']
        # normalized → pixel 좌표 (input_size 기준)
        cx_px = cx * input_size
        cy_px = cy * input_size
        w_px = w * input_size
        h_px = h * input_size
        x_left = cx_px - w_px / 2.0
        y_top = cy_px - h_px / 2.0
        boxes.append([cls_id, float(y_top), float(x_left), float(h_px), float(w_px)])

    if len(boxes) > 0:
        image_filenames.append(img_path)
        labels.append(boxes)

print(f"✅ {len(image_filenames)} images with labels loaded")

# ── 이미지 로드 (ICC 프로파일 제거 후 읽기) ──────────────────────
from PIL import Image, PngImagePlugin
import struct, zlib

def read_png_skip_icc(path):
    """iCCP 청크를 건너뛰고 PNG를 읽는다."""
    with open(path, 'rb') as f:
        signature = f.read(8)
        chunks = []
        while True:
            raw = f.read(8)
            if len(raw) < 8:
                break
            length, ctype = struct.unpack('>I4s', raw)
            data = f.read(length)
            crc = f.read(4)
            if ctype in (b'iCCP', b'tEXt', b'iTXt', b'zTXt'):
                continue  # 큰 메타데이터 청크 스킵
            chunks.append(raw + data + crc)
    import io
    buf = io.BytesIO(signature + b''.join(chunks))
    return Image.open(buf).convert('RGB')

print("\n📷 Loading images...")
images = []
valid_labels = []
skip_count = 0
for img_path, lbl in tqdm(zip(image_filenames, labels), total=len(image_filenames)):
    try:
        img = np.array(read_png_skip_icc(img_path))
    except Exception as e:
        skip_count += 1
        continue
    images.append(img)
    valid_labels.append(lbl)

labels = valid_labels
print(f"✅ {len(images)} images loaded  |  skipped: {skip_count}  |  shape: {images[0].shape}")

In [ ]:
def collate_fn1(batch):
    """배치 collation"""
    samples, cls, box, indices = zip(*batch)
    cls = torch.cat(cls, dim=0)
    box = torch.cat(box, dim=0)
    new_indices = list(indices)
    for i in range(len(indices)):
        new_indices[i] += i
    indices = torch.cat(new_indices, dim=0)
    targets = {'cls': cls, 'box': box, 'idx': indices}
    return torch.stack(samples, dim=0), targets


class custom_dataset(data.Dataset):
    """
    HnE 세포 검출용 YOLO 데이터셋
    - 입력 라벨 : [class_id, y_top, x_left, h, w] pixel 좌표
    - 출력 box  : [cx_norm, cy_norm, w_norm, h_norm] (YOLO 정규화 포맷)
    """
    def __init__(self, images, params, augment, labels):
        self.params   = params
        self.augment  = augment
        self.images   = images
        self.labels   = labels   # [[class, y_top, x_left, h, w], ...] pixel
        self.scale_list = [256, 288, 320, 352, 384, 416, 448, 480, 512]
        self.current_scale = 512
        self.base_size     = 512
        self.n       = len(self.images)
        self.indices = list(range(self.n))

    def __len__(self):
        return len(self.indices)

    def set_scale(self, scale):
        self.current_scale = scale

    # ── 랜덤 크롭 / 패딩 (DINO 노트북과 동일) ───────────────────
    def crop_padding_image(self, image):
        image = image.copy()
        h, w  = image.shape[:2]
        S     = self.base_size
        if min(h, w) > S:
            row_off = random.randint(0, h - S)
            col_off = random.randint(0, w - S)
            image   = image[row_off:row_off + S, col_off:col_off + S]
        else:
            row_off, col_off = 0, 0
            pad = np.ones((S, S, 3), dtype=np.uint8) * 255
            pad[:min(h, S), :min(w, S)] = image[:min(h, S), :min(w, S)]
            image = pad
        return image, row_off, col_off

    # ── 색상 증강 (DINO 노트북과 동일) ──────────────────────────
    def apply_color_augmentation(self, image):
        if random.random() < 0.5:
            f = random.uniform(0.8, 1.2)
            image = np.clip(image * f, 0, 255).astype(np.uint8)
        if random.random() < 0.5:
            f = random.uniform(0.8, 1.2)
            mean = image.mean()
            image = np.clip((image - mean) * f + mean, 0, 255).astype(np.uint8)
        if random.random() < 0.5:
            hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
            hsv[:, :, 0] = np.clip(hsv[:, :, 0] + random.uniform(-10, 10), 0, 179)
            image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
        if random.random() < 0.5:
            hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
            hsv[:, :, 1] = np.clip(hsv[:, :, 1] * random.uniform(0.7, 1.3), 0, 255)
            image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
        if random.random() < 0.3:
            inv_gamma = 1.0 / random.uniform(0.8, 1.2)
            table = np.array([((i / 255.0) ** inv_gamma) * 255
                              for i in range(256)]).astype(np.uint8)
            image = cv2.LUT(image, table)
        if random.random() < 0.3:
            for c in range(3):
                image[:, :, c] = np.clip(
                    image[:, :, c].astype(np.float32) + random.uniform(-10, 10), 0, 255
                ).astype(np.uint8)
        if random.random() < 0.3:
            noise = np.random.normal(0, random.uniform(0, 5), image.shape)
            image = np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)
        if random.random() < 0.2:
            ks = random.choice([3, 5])
            image = cv2.GaussianBlur(image, (ks, ks), 0)
        return image

    def __getitem__(self, index):
        
        index          = self.indices[index]
        original_image = self.images[index].copy()
        raw_labels     = self.labels[index]   # [[class, y_top, x_left, h, w], ...]
        S              = self.base_size

        # ── 1. 크롭/패딩 → 박스 필터링 + 정규화 ─────────────────
        crop_boxes = []
        while len(crop_boxes) < 1:
            image, row_off, col_off = self.crop_padding_image(original_image)
            crop_boxes = []

            for lbl in raw_labels:
                cls_id, y_top, x_left, h, w = lbl
                cy_center = y_top  + h / 2.0   # 행(row) 중심
                cx_center = x_left + w / 2.0   # 열(col) 중심

                # 중심점이 크롭 내에 있어야
                if not (row_off <= cy_center <= row_off + S and
                        col_off <= cx_center <= col_off + S):
                    continue

                new_y = np.clip(y_top  - row_off, 0, S)
                new_x = np.clip(x_left - col_off, 0, S)
                new_h = np.clip(h, 0, S - new_y)
                new_w = np.clip(w, 0, S - new_x)

                if new_h < 1 or new_w < 1:
                    continue

                # YOLO 정규화: [cx_norm, cy_norm, w_norm, h_norm]
                cx_norm = (new_x + new_w / 2.0) / S
                cy_norm = (new_y + new_h / 2.0) / S
                w_norm  = new_w / S
                h_norm  = new_h / S
                crop_boxes.append([int(cls_id), cx_norm, cy_norm, w_norm, h_norm])

        boxes = np.array(crop_boxes, dtype=np.float32)  # (N, 5)

        # ── 2. 기하 증강 ─────────────────────────────────────────
        if self.augment:
            # Vertical flip: cy → 1 - cy
            if random.random() < self.params['flip_ud']:
                image = np.flipud(image).copy()
                boxes[:, 2] = 1 - boxes[:, 2]

            # Horizontal flip: cx → 1 - cx
            if random.random() < self.params['flip_lr']:
                image = np.fliplr(image).copy()
                boxes[:, 1] = 1 - boxes[:, 1]

            # 90° 회전 (DINO 노트북과 동일)
            if random.random() < 0.3:
                k = random.randint(1, 3)
                image = np.rot90(image, k).copy()
                for _ in range(k):
                    cx, cy = boxes[:, 1].copy(), boxes[:, 2].copy()
                    bw, bh = boxes[:, 3].copy(), boxes[:, 4].copy()
                    boxes[:, 1] = cy        # new cx = old cy
                    boxes[:, 2] = 1 - cx   # new cy = 1 - old cx
                    boxes[:, 3] = bh       # new w  = old h
                    boxes[:, 4] = bw       # new h  = old w

            # 색상 증강
            image = self.apply_color_augmentation(image)

        # ── 3. 멀티스케일 리사이즈 ───────────────────────────────
        size = self.current_scale
        if size != S:
            image = cv2.resize(image, (size, size))
            # 정규화 좌표는 resize에 무관

        # ── 4. HWC → CHW ─────────────────────────────────────────
        image   = image.transpose((2, 0, 1))
        cls_arr = boxes[:, 0].astype(np.float32)
        box_arr = boxes[:, 1:5].astype(np.float32)   # [cx, cy, w, h] norm
        nl      = len(box_arr)

        return (torch.from_numpy(image),
                torch.from_numpy(cls_arr),
                torch.from_numpy(box_arr),
                torch.zeros(nl))


# ── Train / Val split ────────────────────────────────────────────
patch_train, patch_test, label_train, label_test = train_test_split(
    images, labels, test_size=0.1, random_state=242, shuffle=True
)

train_dataset = custom_dataset(patch_train, params, augment=True,  labels=label_train)
val_dataset   = custom_dataset(patch_test,  params, augment=False, labels=label_test)

# ── 클래스 분포 분석 ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("📊 클래스 분포 분석")
print("=" * 70)

total_counts = np.zeros(num_classes, dtype=np.float64)
for lbl_list in label_train:
    for lbl in lbl_list:
        total_counts[int(lbl[0])] += 1

print("\n클래스별 샘플 수:")
total_n = total_counts.sum()
for i in range(num_classes):
    name = class_names[i]
    print(f"  [{i}] {name:20s}: {int(total_counts[i]):6d}개  ({total_counts[i]/total_n*100:.2f}%)")

# ── sqrt 역빈도 WeightedRandomSampler ────────────────────────────
# 72배 불균형 → sqrt로 완화 (DINO 노트북과 동일)
cls_w = 1.0 / np.sqrt(total_counts + 1.0)

print(f"\n  상대 샘플링 비율 (class0 기준):")
for i in range(num_classes):
    ratio = cls_w[i] / cls_w[0]
    print(f"  [{i}] {class_names[i]:20s}: {ratio:.1f}x")

sample_weights = []
for lbl_list in label_train:
    if len(lbl_list) == 0:
        sample_weights.append(1.0)
    else:
        sample_weights.append(float(max(cls_w[int(lbl[0])] for lbl in lbl_list)))

sample_weights = torch.tensor(sample_weights, dtype=torch.double)
sampler = torch.utils.data.WeightedRandomSampler(
    weights=sample_weights, num_samples=len(sample_weights), replacement=True
)

# 🎯 Recall 개선: 클래스 가중치를 0.7 power로 완화 (너무 강한 가중치 방지)
# inverse frequency^0.7 → 소수 클래스에 더 많은 기회, 하지만 과도하지 않게
cls_loss_w = total_n / (num_classes * (total_counts + 1.0))
cls_loss_w = np.power(cls_loss_w, 0.7)  # 완화
min_w = cls_loss_w.min()
params['class_weights'] = (cls_loss_w / min_w).tolist()
print(f"\n✅ Loss class_weights (power=0.7): {[f'{w:.2f}' for w in params['class_weights']]}")
print("=" * 70)

# ── DataLoader ───────────────────────────────────────────────────
batch_size = 16

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size,
    num_workers=4, collate_fn=collate_fn1, drop_last=True
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=4, collate_fn=collate_fn1, drop_last=False
)

print(f"\n📦 DataLoaders ready:")
print(f"  Train batches : {len(train_loader)}  (WeightedRandomSampler 적용)")
print(f"  Val batches   : {len(val_loader)}")


In [ ]:
def visualize_sample_with_overlay(dataset, index=0):
    image_tensor, cls_tensor, box_tensor, _ = dataset[index]

    image  = image_tensor.numpy().transpose(1, 2, 0)
    cls    = cls_tensor.numpy()
    boxes  = box_tensor.numpy()   # [cx_norm, cy_norm, w_norm, h_norm]
    height, width = image.shape[:2]

    colors = ['red', 'limegreen', 'yellow', 'magenta', 'dodgerblue', 'orange']

    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(image.astype(np.uint8))

    for i in range(len(boxes)):
        cx_norm, cy_norm, w_norm, h_norm = boxes[i]
        cx_px = cx_norm * width
        cy_px = cy_norm * height
        w_px  = w_norm  * width
        h_px  = h_norm  * height
        x1 = cx_px - w_px / 2
        y1 = cy_px - h_px / 2

        class_id = int(cls[i])
        color = colors[class_id % len(colors)]
        rect = patches.Rectangle((x1, y1), w_px, h_px,
                                  linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)

    ax.set_title(f'Sample {index} | Total Objects: {len(boxes)}', fontsize=12)
    ax.axis('off')

    legend_elements = [
        patches.Patch(color=colors[i], label=f'{class_names[i]} ({int((cls==i).sum())})')
        for i in range(num_classes)
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=3,
               bbox_to_anchor=(0.5, -0.05), fontsize=9)

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15)
    plt.show()

    print(f"\n📊 Sample {index}:")
    print(f"  이미지 크기 : {width} x {height}")
    print(f"  총 객체 수  : {len(boxes)}")
    for i in range(num_classes):
        count = int((cls == i).sum())
        if count > 0:
            print(f"  [{i}] {class_names[i]:20s}: {count}개")

visualize_sample_with_overlay(train_dataset, index=0)


In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, LinearLR, SequentialLR

# 클래스 가중치 적용 가능한 사용자 정의 ComputeLoss 클래스
class WeightedComputeLoss:
    def __init__(self, model, params):
        if hasattr(model, 'module'):
            model = model.module

        device = next(model.parameters()).device
        m = model.head  # Head() module

        self.params = params
        self.stride = m.stride
        self.nc = m.nc
        self.no = m.no
        self.reg_max = m.ch
        self.device = device

        self.box_loss = util.BoxLoss(m.ch - 1).to(device)
        
        # 클래스 가중치 적용
        if 'class_weights' in params and params['class_weights'] is not None:
            class_weights = torch.tensor(params['class_weights'], dtype=torch.float32).to(device)
            print(f"🎯 클래스 가중치 적용됨: {params['class_weights']}")
        else:
            class_weights = None
            print("⚠️  클래스 가중치 없음 - 균등 가중치 사용")
        
        self.cls_loss = torch.nn.BCEWithLogitsLoss(pos_weight=class_weights, reduction='none') if class_weights is not None else torch.nn.BCEWithLogitsLoss(reduction='none')
        
        # 🎯 Recall 개선: top_k 증가 (10→20), alpha 낮춤 (0.5→0.3)
        # top_k: positive sample 수 증가 → 더 많은 객체 학습
        # alpha: 분류 점수보다 위치 중요도 증가 → detection 능력 향상
        self.assigner = util.Assigner(nc=self.nc, top_k=20, alpha=0.3, beta=6.0)
        self.project = torch.arange(m.ch, dtype=torch.float, device=device)

    def box_decode(self, anchor_points, pred_dist):
        b, a, c = pred_dist.shape
        pred_dist = pred_dist.view(b, a, 4, c // 4)
        pred_dist = pred_dist.softmax(3)
        pred_dist = pred_dist.matmul(self.project.type(pred_dist.dtype))
        lt, rb = pred_dist.chunk(2, -1)
        x1y1 = anchor_points - lt
        x2y2 = anchor_points + rb
        return torch.cat(tensors=(x1y1, x2y2), dim=-1)

    def __call__(self, outputs, targets):
        x = torch.cat([i.view(outputs[0].shape[0], self.no, -1) for i in outputs], dim=2)
        pred_distri, pred_scores = x.split(split_size=(self.reg_max * 4, self.nc), dim=1)

        pred_scores = pred_scores.permute(0, 2, 1).contiguous()
        pred_distri = pred_distri.permute(0, 2, 1).contiguous()

        data_type = pred_scores.dtype
        batch_size = pred_scores.shape[0]
        input_size = torch.tensor(outputs[0].shape[2:], device=self.device, dtype=data_type) * self.stride[0]
        anchor_points, stride_tensor = util.make_anchors(outputs, self.stride, offset=0.5)

        idx = targets['idx'].view(-1, 1)
        cls = targets['cls'].view(-1, 1)
        box = targets['box']

        targets = torch.cat((idx, cls, box), dim=1).to(self.device)
        if targets.shape[0] == 0:
            gt = torch.zeros(batch_size, 0, 5, device=self.device)
        else:
            i = targets[:, 0]
            _, counts = i.unique(return_counts=True)
            counts = counts.to(dtype=torch.int32)
            gt = torch.zeros(batch_size, counts.max(), 5, device=self.device)
            for j in range(batch_size):
                matches = i == j
                n = matches.sum()
                if n:
                    gt[j, :n] = targets[matches, 1:]
            x = gt[..., 1:5].mul_(input_size[[1, 0, 1, 0]])
            y = torch.empty_like(x)
            dw = x[..., 2] / 2  # half-width
            dh = x[..., 3] / 2  # half-height
            y[..., 0] = x[..., 0] - dw  # top left x
            y[..., 1] = x[..., 1] - dh  # top left y
            y[..., 2] = x[..., 0] + dw  # bottom right x
            y[..., 3] = x[..., 1] + dh  # bottom right y
            gt[..., 1:5] = y
        gt_labels, gt_bboxes = gt.split((1, 4), 2)
        mask_gt = gt_bboxes.sum(2, keepdim=True).gt_(0)

        pred_bboxes = self.box_decode(anchor_points, pred_distri)
        assigned_targets = self.assigner(pred_scores.detach().sigmoid(),
                                         (pred_bboxes.detach() * stride_tensor).type(gt_bboxes.dtype),
                                         anchor_points * stride_tensor, gt_labels, gt_bboxes, mask_gt)
        target_bboxes, target_scores, fg_mask = assigned_targets

        target_scores_sum = max(target_scores.sum(), 1)

        # 클래스 가중치가 적용된 Classification Loss
        loss_cls = self.cls_loss(pred_scores, target_scores.to(data_type)).sum() / target_scores_sum

        # Box loss
        loss_box = torch.zeros(1, device=self.device)
        loss_dfl = torch.zeros(1, device=self.device)
        if fg_mask.sum():
            target_bboxes /= stride_tensor
            loss_box, loss_dfl = self.box_loss(pred_distri,
                                               pred_bboxes,
                                               anchor_points,
                                               target_bboxes,
                                               target_scores,
                                               target_scores_sum, fg_mask)

        loss_box *= self.params['box']  # box gain
        loss_cls *= self.params['cls']  # cls gain
        loss_dfl *= self.params['dfl']  # dfl gain

        return loss_box, loss_cls, loss_dfl

# 모델 및 파라미터 준비 (일반 YOLOv11m)
model = nn.yolo_v11_m(len(params['names'])).to(device)
optimizer = torch.optim.AdamW(util.set_params(model, params['weight_decay']),
                             lr=1e-3, 
                             betas=(0.9, 0.999),
                             weight_decay=params['weight_decay'])

# 클래스 가중치가 적용된 사용자 정의 ComputeLoss 사용
criterion = WeightedComputeLoss(model, params)

cosine_scheduler = CosineAnnealingWarmRestarts(
    optimizer,
    T_0=10,  # 10 에폭마다 재시작
    T_mult=2,  # 재시작 주기를 2배씩 증가
    eta_min=float(params['min_lr'])
)
# 데이터셋 및 데이터로드 (안전한 함수 사용)
batch_size = 16
# 안전하게 데이터로더 생성하는 함수
def create_safe_loader(dataset, batch_size, is_train=True):
    """
    배치 크기에 맞게 데이터셋을 조정하여 안전하게 데이터로더를 생성하는 함수
    """
    dataset_size = len(dataset)
    
    # 배치 크기가 데이터셋 크기보다 큰 경우 배치 크기 조정
    if dataset_size < batch_size:
        print(f"경고: 데이터셋 크기({dataset_size})가 배치 크기({batch_size})보다 작습니다. 배치 크기를 {dataset_size}로 조정합니다.")
        actual_batch_size = max(1, dataset_size)
    else:
        actual_batch_size = batch_size
    
    # 데이터셋이 배치 크기로 나누어 떨어지는지 확인
    if dataset_size % actual_batch_size != 0:
        print(f"참고: 데이터셋 크기({dataset_size})가 배치 크기({actual_batch_size})로 나누어 떨어지지 않습니다.")
        print(f"마지막 배치는 {dataset_size % actual_batch_size}개의 샘플을 포함합니다.")
    
    # 데이터로더 생성
    loader = torch.utils.data.DataLoader(
        dataset, 
        batch_size=actual_batch_size, 
        shuffle=is_train,
        num_workers=4,
        collate_fn=collate_fn1,
        drop_last=(not is_train)  # 훈련 시에는 마지막 배치 유지, 검증 시에는 마지막 배치 제외
    )
    
    return loader, actual_batch_size
# 안전하게 데이터로더 생성
loader, train_batch_size = create_safe_loader(train_dataset, batch_size, is_train=True)
val_loader, val_batch_size = create_safe_loader(val_dataset, batch_size, is_train=False)

print(f"최종 훈련 배치 크기: {train_batch_size}")
print(f"최종 검증 배치 크기: {val_batch_size}")
print(f"\n🎯 Recall 개선을 위한 학습 파라미터 변경:")
print(f"  ✅ Assigner top_k: 20 (기본 10 → 더 많은 positive sample)")
print(f"  ✅ Assigner alpha: 0.3 (기본 0.5 → 분류보다 위치 중시)")
print(f"  ✅ Loss 가중치: box={params['box']}, cls={params['cls']}")
print(f"  ✅ 학습률: min={params['min_lr']}, max={params['max_lr']}")
print(f"  ✅ 클래스 가중치: power=0.7 완화")


In [ ]:
from utils.valid import compute_point_label_metrics_single
from utils.valid import visualize_ground_truth_and_prediction_separately_detail_single
from utils.valid import plot_training_progress

train_losses        = []
val_det_recalls     = []
val_cls_accs        = []
val_macro_f1s       = []
val_macro_precisions = []
val_macro_recalls   = []
val_class_stats_history = []

epochs     = 1000
scale_list = [512]

save_dir = '../../model/precise_BC_cell_scoring/her2_yolov11/'
os.makedirs(save_dir, exist_ok=True)

# 체크포인트 불러오기 (재개 시 주석 해제)
# checkpoint_path = os.path.join(save_dir, 'last_model.pt')
# if os.path.exists(checkpoint_path):
#     checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
#     model.load_state_dict(checkpoint['model_state_dict'])

best_macro_Overall_Recall = 0
accumulate = max(round(64 / batch_size), 1)
amp_scale  = torch.amp.GradScaler()

print(f"Gradient accumulation steps: {accumulate}")

for epoch in range(epochs):
    model.train()

    avg_box_loss = util.AverageMeter()
    avg_cls_loss = util.AverageMeter()
    avg_dfl_loss = util.AverageMeter()

    train_pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                      desc=f'Epoch {epoch+1}/{epochs} Training')

    optimizer.zero_grad()
    current_scale = random.choice(scale_list)
    train_dataset.set_scale(current_scale)
    # sampler가 있으므로 학습 루프 안에서 DataLoader 재생성 불필요
    # (scale 변경은 set_scale로 처리)

    for i, (torch_images, targets) in train_pbar:
        step = i + len(train_loader) * epoch
        torch_images = torch_images.to(device).float() / 255.

        with torch.amp.autocast('cuda'):
            outputs = model(torch_images)
            loss_box, loss_cls, loss_dfl = criterion(outputs, targets)

        avg_box_loss.update(loss_box.item(), torch_images.size(0))
        avg_cls_loss.update(loss_cls.item(), torch_images.size(0))
        avg_dfl_loss.update(loss_dfl.item(), torch_images.size(0))

        loss_box *= batch_size
        loss_cls *= batch_size
        loss_dfl *= batch_size
        total_loss = loss_box + loss_cls + loss_dfl

        amp_scale.scale(total_loss).backward()

        if step % accumulate == 0:
            amp_scale.step(optimizer)
            amp_scale.update()
            optimizer.zero_grad()

        torch.cuda.synchronize()

        memory = f'{torch.cuda.memory_reserved() / 1E9:.4g}G'
        s = (f'Memory: {memory} | Box: {avg_box_loss.avg:.3f} | '
             f'Cls: {avg_cls_loss.avg:.3f} | DFL: {avg_dfl_loss.avg:.3f}')
        train_pbar.set_description(f'Epoch {epoch+1}/{epochs} | {s}')

    avg_train_loss = avg_box_loss.avg + avg_cls_loss.avg + avg_dfl_loss.avg
    train_losses.append(avg_train_loss)
    val_dataset.set_scale(512)

    # ── Validation ──────────────────────────────────────────────
    try:
        point_metrics = compute_point_label_metrics_single(
            model, val_loader, device, params, distance_threshold=16
        )
        detection_recall = point_metrics.get('detection_recall', 0)
        cls_accuracy     = point_metrics.get('classification_accuracy', 0)
        macro_precision  = point_metrics.get('macro_precision', 0)
        macro_recall     = point_metrics.get('macro_recall', 0)
        macro_f1         = point_metrics.get('macro_f1', 0)
        overall_recall   = point_metrics.get('overall_recall', 0)
        class_stats      = point_metrics.get('class_stats', {})

        val_det_recalls.append(detection_recall)
        val_cls_accs.append(cls_accuracy)
        val_macro_f1s.append(macro_f1)
        val_macro_precisions.append(macro_precision)
        val_macro_recalls.append(macro_recall)
        val_class_stats_history.append(class_stats)

        print(f"\nEpoch {epoch+1}/{epochs}:")
        print(f"  Train Loss — Box: {avg_box_loss.avg:.4f} | Cls: {avg_cls_loss.avg:.4f} | DFL: {avg_dfl_loss.avg:.4f}")
        print(f"  Detection Recall       : {detection_recall:.4f}")
        print(f"  Classification Accuracy: {cls_accuracy:.4f}")
        print(f"  Macro Precision        : {macro_precision:.4f}")
        print(f"  Macro Recall           : {macro_recall:.4f}")
        print(f"  Macro F1               : {macro_f1:.4f} ⭐")
        print(f"  Overall Recall         : {overall_recall:.4f}")
    except Exception as e:
        print(f"Validation 오류: {e}")
        detection_recall = cls_accuracy = macro_precision = macro_recall = macro_f1 = overall_recall = 0
        class_stats = {}

    # ── Checkpoint ──────────────────────────────────────────────
    ckpt_data = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'amp_scale_state_dict': amp_scale.state_dict(),
        'train_loss': avg_train_loss,
        'box_loss': avg_box_loss.avg, 'cls_loss': avg_cls_loss.avg, 'dfl_loss': avg_dfl_loss.avg,
        'detection_recall': detection_recall, 'classification_accuracy': cls_accuracy,
        'macro_precision': macro_precision, 'macro_recall': macro_recall,
        'macro_f1': macro_f1, 'overall_recall': overall_recall,
        'class_stats': class_stats, 'params': params,
    }
    torch.save(ckpt_data, os.path.join(save_dir, 'last_model.pt'))

    if overall_recall > best_macro_Overall_Recall:
        best_macro_Overall_Recall = overall_recall
        torch.save(ckpt_data, os.path.join(save_dir, 'best_model.pt'))
        print(f"🎉 새로운 베스트! Overall Recall: {overall_recall:.4f}")

    # ── 시각화 (10 epoch마다) ────────────────────────────────────
    if (epoch + 1) % 10 == 0:
        try:
            sample_idx = random.randint(0, len(val_dataset) - 1)
            visualize_ground_truth_and_prediction_separately_detail_single(
                model, val_dataset, idx=sample_idx,
                epoch=epoch + 1, save_dir=save_dir
            )
        except Exception as e:
            print(f"시각화 오류: {e}")

    # ── 학습 곡선 (100 epoch마다) ─────────────────────────────────
    if (epoch + 1) % 100 == 0:
        try:
            plot_training_progress(
                train_losses, val_det_recalls, val_cls_accs, val_macro_precisions,
                val_macro_recalls, val_macro_f1s, epoch + 1, save_dir,
                class_stats_history=val_class_stats_history
            )
        except Exception as e:
            print(f"그래프 오류: {e}")

print("🎯 학습 완료!")
print(f"  Best Overall Recall : {best_macro_Overall_Recall:.4f}")
print(f"  저장 경로           : {save_dir}")
